# Import libraries

In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

import time
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np

import tensorflow as tf
from keras.layers import Activation, BatchNormalization, Conv1D
from keras import initializers

import matplotlib.patches as patches

import gc


import lasio

import pandas as pd

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

tf.autograph.set_verbosity(0)


2024-06-14 10:14:37.401718: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-14 10:14:37.401791: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-14 10:14:37.447262: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-06-14 10:14:37.554906: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-14 10:14:39.174466: W tensorflow/compiler/tf2

# Load data

In [2]:
seis_mature= np.load('../data_decatur/processed/mature_block.npy')
phi_mature = np.load('../data_decatur/processed/mature_block_porosity.npy')

seis_exploration = np.load('../data_decatur/processed/exploration_block.npy')
phi_exploration = np.load('../data_decatur/processed/exploration_block_porosity.npy')

In [3]:
epocas=100
given_seed=13

# Graficos

In [1]:
def visualizacion_resultados(history):
    epochs = [i for i in range(epocas)]

    train_acc = history.history['r2_nicolas']
    train_loss = history.history['loss']
    train_lr = history.history['lr']
    val_acc = history.history['val_r2_nicolas']
    val_loss = history.history['val_loss']
    

    fig, ax = plt.subplots(1,3)
    fig.set_size_inches(16,7)

    ax[0].plot(epochs, train_acc, 'go-', label='accuracy-train')
    ax[0].plot(epochs, val_acc, 'ro-', label='accuracy-val')
    ax[0].set_title('Accuracy train')
    ax[0].legend()
    ax[0].set_xlabel('epochs')
    ax[0].set_ylabel('accuracy')
    ax[0].set_ylim(-1,1)

    ax[1].plot(epochs, train_loss, 'go-', label='loss-train')
    ax[1].plot(epochs, val_loss, 'ro-', label='loss-val')
    ax[1].set_title('Loss train')
    ax[1].legend()
    ax[1].set_xlabel('epochs')
    ax[1].set_ylabel('loss')
    
    ax[2].plot(epochs, train_lr, 'go-', label='lr-train')
    ax[2].set_title('Learning Rate train')
    ax[2].legend()
    ax[2].set_xlabel('epochs')
    ax[2].set_ylabel('Learning Rate')
    
    fig.savefig('./plots/training.png')

    plt.show()

# Shape and Statistics

In [5]:
depth_start = 4530
depth_end = 7010
depth_step = 20
depth_values = np.arange(depth_start, depth_end , depth_step)
num_ticks = 6  # Adjust the number of ticks as needed
depth_indices = np.linspace(0, len(depth_values) - 1, num_ticks, dtype=int)

# Normalizacion de Datos

In [6]:
def min_max_scale(x, min, max):

  x_std = (x - min) / (max - min)
  x_scaled = x_std * 2 - 1
  return x_scaled

def inverse_min_max_scale(x, min, max):

  x_normalized = (x + 1) / 2
  x_unscaled = x_normalized * (max - min) + min
  return x_unscaled

In [7]:
phi_mature[phi_mature<0] = 0.0001
phi_max=np.max(phi_mature) #can also take 1 or critical porosity (0.4)
phi_min=np.min(phi_mature) #can also take 0

In [8]:
phi_scaled = min_max_scale(phi_mature, min= phi_min, max=phi_max)
seis_normalized = (seis_mature - np.min(seis_mature))/(np.max(seis_mature)-np.min(seis_mature))

In [9]:
def naive_model():
    """El input shape de tal forma sería entonces (1, 246) ya que son trazas sismicas, 2000 de ellas"""

    model = tf.keras.models.Sequential()
        #probar a poner capa densa al comienzo y luego pasar por la convolución
    input_final = tf.keras.Input(shape=(1,124))
    model.add(input_final)
    #model.add(tf.keras.layers.Reshape((-1,1)))
    
    model.add(Conv1D(filters=128, kernel_size=3, strides=1, padding='same',kernel_initializer=initializers.he_uniform(seed=given_seed), bias_initializer='zeros')) 
    model.add(Activation('tanh'))
    
    model.add(Conv1D(filters=256, kernel_size=3, strides=1, padding='same',kernel_initializer=initializers.he_uniform(seed=given_seed), bias_initializer='zeros')) 
    model.add(Activation('tanh'))
    
    model.add(Conv1D(filters=512, kernel_size=3, strides=1, padding='same',kernel_initializer=initializers.he_uniform(seed=given_seed), bias_initializer='zeros')) 
    model.add(Activation('tanh'))
    
    model.add(Conv1D(filters=1024, kernel_size=3, strides=1, padding='same',kernel_initializer=initializers.he_uniform(seed=given_seed), bias_initializer='zeros')) 
    model.add(Activation('tanh'))
    model.add(tf.keras.layers.Dropout(rate=0.3))
    

    model.add(tf.keras.layers.Flatten())
 
    
    model.add(tf.keras.layers.Dense(units=124,kernel_initializer=initializers.he_uniform(seed=given_seed), bias_initializer='zeros'))
    model.add(Activation('tanh'))

    return model

In [10]:
def r2_nicolas(Y_val_final, phi_pred):
    sum_squares_residuals = tf.reduce_sum((Y_val_final - phi_pred) ** 2)
    sum_squares = tf.reduce_sum((Y_val_final - tf.reduce_mean(Y_val_final)) ** 2)
    R2 = 1 - sum_squares_residuals / sum_squares
    return R2

optimizer = tf.keras.optimizers.Adam()


2024-06-14 10:14:42.158814: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-14 10:14:42.372724: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-14 10:14:42.372776: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-14 10:14:42.377148: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-14 10:14:42.377225: I external/local_xla/xla/stream_executor

In [11]:
#callback_early = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, mode='auto')
#checkpoint = tf.keras.callbacks.ModelCheckpoint('model_naive.h5', 
#                                                monitor='val_r2_nicolas', 
#                                                mode='max', 
#                                                verbose=0, 
#                                                save_weights_only=True)

callback_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='r2_nicolas', 
                                                   factor=0.5, 
                                                   patience=50, 
                                                   min_lr=0.000001, 
                                                    verbose=0, )

# Experimento

In [12]:
results = pd.DataFrame(columns=['Porcentaje', 'Iteraciones', 'R2', 'Loss', 'Time'])

In [13]:
porcentajes = [75]
for porcentaje_entrenamiento in porcentajes:
    print(f'Entrenamiento para {porcentaje_entrenamiento}% de los datos')
    for iter in range(10):
        print(f'Entrenamiento {iter+1}')
        porcentaje_validacion= 1

        train_wells = int((seis_normalized.shape[0] * seis_normalized.shape[1]) * (porcentaje_entrenamiento/100))
        val_wells = int((seis_normalized.shape[0] * seis_normalized.shape[1]) * (porcentaje_validacion/100))

        coords_train = np.zeros((train_wells, 2))
        coords_val = np.zeros((val_wells, 2))


        # Create a mask to keep track of which indices have been used
        mask = np.ones(seis_normalized.shape[:2], dtype=bool)

        def extract_traces(numer_of_wells, train=True):
            x_seismic = []
            y_porosity = []
            for _ in range(numer_of_wells):
                random_il = np.random.randint(0, seis_normalized.shape[0])
                random_xl = np.random.randint(0, seis_normalized.shape[1])
                
                # Ensure we pick an unused index
                while not mask[random_il, random_xl]:
                    random_il = np.random.randint(0, seis_normalized.shape[0])
                    random_xl = np.random.randint(0, seis_normalized.shape[1])
                    
                if train:
                    coords_train[_] = np.array([random_il, random_xl])
                else:
                    coords_val[_] = np.array([random_il, random_xl])
                
                X_chosen = seis_normalized[random_il, random_xl, :]
                Y_chosen = phi_scaled[random_il, random_xl, :]
                
                x_seismic.append(np.expand_dims(X_chosen, axis=0))
                y_porosity.append(np.expand_dims(Y_chosen, axis=0))
                
                # Mark this index as used
                mask[random_il, random_xl] = False

            x_seismic = np.concatenate(x_seismic, axis=0)
            y_porosity = np.concatenate(y_porosity, axis=0)
            
            return x_seismic, y_porosity
            
        X_train, Y_train = extract_traces(train_wells)
        X_val, Y_val = extract_traces(val_wells, train=False)
            
        # Remaining data for test
        X_test = seis_normalized[mask]
        Y_test = phi_scaled[mask]
        
        X_train_final = np.expand_dims(X_train, axis=1)
        X_test_final = np.expand_dims(X_test, axis=1)

        X_val_final = np.expand_dims(X_val, axis=1)
        
        tf.keras.backend.clear_session()
        gc.collect()
        
        model_phi = naive_model()
        model_phi.build((None,124,1))
        
        optimizer = tf.keras.optimizers.Adam()

        model_phi.compile(optimizer=optimizer,
                    loss='mse',
                    metrics=r2_nicolas)
        #os.remove("model_naive.h5")
        
        #Training
        start_time = time.time()

        model_phi.fit(x=X_train_final, 
                                y=Y_train,
                                epochs=epocas,
                                batch_size=100,
                                callbacks=[callback_lr],
                                validation_data=(X_val_final, Y_val), 
                                verbose=0                        
                                )
        # EVALUACION DE LA RED 
        end_time = time.time()
        # Calculate the elapsed time
        elapsed_time = np.round(end_time - start_time)
        loss, accuracy = model_phi.evaluate(X_test_final, Y_test)
        
        current_results = {'Porcentaje':[porcentaje_entrenamiento], 'Iteraciones':[iter+1], 'R2':[accuracy], 'Loss':[loss], 'Time':[elapsed_time]}
        print(current_results)
        results = pd.concat([results, pd.DataFrame(current_results)])

        # PREDICCION DE LA RED
        """seis_normalized_exploration = (seis_exploration - np.min(seis_exploration))/(np.max(seis_exploration)-np.min(seis_exploration))

        X_exploracion = seis_normalized_exploration.reshape(-1, 124)

        X_exploracion = np.expand_dims(X_exploracion, axis=1)
        phi_pred_exploracion = model_phi.predict(X_exploracion)
        phi_pred_exploracion = inverse_min_max_scale(phi_pred_exploracion, phi_min, phi_max)
        phi_pred_exploracion = phi_pred_exploracion.reshape(183, 370, 124)
        fig, ax = plt.subplots(1,3, figsize = (15, 4))

        fig.suptitle('Inline 84 Wells CCS1 VW1')

        im1 = ax[0].imshow(seis_exploration[84-13,:,:].T, cmap='Greys')
        ax[0].set_title('Seismic')
        ax[0].set_aspect('auto')
        ax[0].set_yticks(depth_indices)
        ax[0].set_yticklabels(depth_values[depth_indices])
        fig.colorbar(im1, ax=ax[0], shrink=1)

        im2 = ax[1].imshow(phi_pred_exploracion[84-13,:,:].T, vmin=0, vmax=0.3, cmap='jet')
        ax[1].set_title('Estimated Porosity')
        ax[1].set_aspect('auto')
        ax[1].set_yticks(depth_indices)
        ax[1].set_yticklabels(depth_values[depth_indices])
        fig.colorbar(im2, ax=ax[1], shrink=1)

        im3 = ax[2].imshow(phi_exploration[84-13,:,:].T, vmin=0, vmax=0.3, cmap='jet')
        ax[2].set_title('Ground Truth Porosity')
        ax[2].set_aspect('auto')
        ax[2].set_yticks(depth_indices)
        ax[2].set_yticklabels(depth_values[depth_indices])
        fig.colorbar(im3, ax=ax[2], shrink=1)

        fig.tight_layout()
        fig.savefig(f"./plots/section_predicted_inline_{porcentaje_entrenamiento}_of_data.pdf", format="pdf", bbox_inches="tight")

        plt.show()"""
        
        # Clear model explicitly
        del model_phi

        # Force garbage collection again to free up memory
        gc.collect()
        
        
        

Entrenamiento para 75% de los datos
Entrenamiento 1


2024-06-14 10:14:46.902540: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-06-14 10:14:47.134417: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-06-14 10:14:47.293117: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-06-14 10:14:48.439573: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f9ba0997d00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-06-14 10:14:48.439617: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1050, Compute Capability 6.1
2024-06-14 10:14:48.458263: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1718378088.615535   20003 device_compiler.

1155/1155 [==============================] - 6s 5ms/step - loss: 0.0020 - r2_nicolas: 0.9866
{'Porcentaje': [75], 'Iteraciones': [1], 'R2': [0.9865620136260986], 'Loss': [0.0019792895764112473], 'Time': [1163.0]}
Entrenamiento 2


/tmp/ipykernel_19898/3458641220.py:93: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame(current_results)])


1155/1155 [==============================] - 6s 5ms/step - loss: 0.0020 - r2_nicolas: 0.9867
{'Porcentaje': [75], 'Iteraciones': [2], 'R2': [0.9867423176765442], 'Loss': [0.001954643987119198], 'Time': [1204.0]}
Entrenamiento 3
1155/1155 [==============================] - 6s 5ms/step - loss: 0.0020 - r2_nicolas: 0.9865
{'Porcentaje': [75], 'Iteraciones': [3], 'R2': [0.986526608467102], 'Loss': [0.001985247014090419], 'Time': [1208.0]}
Entrenamiento 4
1155/1155 [==============================] - 6s 5ms/step - loss: 0.0020 - r2_nicolas: 0.9867
{'Porcentaje': [75], 'Iteraciones': [4], 'R2': [0.9867223501205444], 'Loss': [0.0019571445882320404], 'Time': [1232.0]}
Entrenamiento 5
1155/1155 [==============================] - 6s 5ms/step - loss: 0.0020 - r2_nicolas: 0.9867
{'Porcentaje': [75], 'Iteraciones': [5], 'R2': [0.986679196357727], 'Loss': [0.0019627215806394815], 'Time': [1236.0]}
Entrenamiento 6
1155/1155 [==============================] - 6s 5ms/step - loss: 0.0020 - r2_nicolas: 0.

In [14]:
results.to_csv('training_results_pt3.csv', index=False)
results

,Porcentaje,Iteraciones,R2,Loss,Time
0,75,1,0.986562,0.001979,1163.0
0,75,2,0.986742,0.001955,1204.0
0,75,3,0.986527,0.001985,1208.0
0,75,4,0.986722,0.001957,1232.0
0,75,5,0.986679,0.001963,1236.0
0,75,6,0.986530,0.001986,1248.0
0,75,7,0.986865,0.001938,1254.0
0,75,8,0.986708,0.001958,1253.0
0,75,9,0.986828,0.001942,1276.0
0,75,10,0.986815,0.001945,1274.0


# Entrenamiento

In [15]:
r2_experiments_val

NameError: name 'r2_experiments_val' is not defined

In [ ]:
print(history_early.history.keys())

In [ ]:
visualizacion_resultados(history_early)

In [ ]:
loss, accuracy = model_phi.evaluate(X_val_final, Y_val)

# Test del Modelo

In [ ]:
# load the saved model
model_phi.load_weights('model_naive.h5')

In [ ]:
X_test_final.shape

In [ ]:
phi_pred = model_phi.predict(X_test_final)

In [ ]:
phi_pred_true = inverse_min_max_scale(phi_pred, phi_min, phi_max)

In [ ]:
depth = np.arange(0,124)

In [ ]:
phi_pred_true.shape

In [ ]:
phi_pred_true[1,:].shape

In [ ]:
plt.plot(phi_pred_true[1,:])
plt.plot(phi_mature[1,1,:])

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(12, 8))
for i in range(0,4):
    il_number = np.random.randint(0,50)
    xl_number = np.random.randint(0,50)
    one_log_pred = phi_pred_true[xl_number,:]
    one_log_true = phi_mature[il_number,xl_number,:]
    ax[i].plot(one_log_true, depth, label='Phie True')
    ax[i].plot(one_log_pred, depth,label='Phi pred')

    ax[i].invert_yaxis()  # This will flip the y-axis
    ax[i].set_title(f'well Number #{xl_number}')
plt.legend()

plt.show()


In [ ]:
fig.savefig("./plots/predictions.png")

## Bloque Maduro

In [ ]:
seis_normalized_mature = (seis_mature - np.min(seis_mature))/(np.max(seis_mature)-np.min(seis_mature))
X_maduro = seis_normalized_mature.reshape(-1, 124)
X_maduro = np.expand_dims(X_maduro, axis=1)
phi_pred_maduro = model_phi.predict(X_maduro)
phi_pred_maduro = inverse_min_max_scale(phi_pred_maduro, phi_min, phi_max)

In [ ]:
seis_normalized_mature.shape

In [ ]:
phi_pred_maduro = phi_pred_maduro.reshape(183, 841, 124)

In [ ]:
fig, ax = plt.subplots(1,3, figsize = (15, 4))

fig.suptitle('Random Xline Mature Block')

im1 = ax[0].imshow(seis_mature[:,100,:].T, cmap='Greys')
ax[0].set_title('Seismic')
ax[0].set_aspect('auto')
ax[0].set_yticks(depth_indices)
ax[0].set_yticklabels(depth_values[depth_indices])
fig.colorbar(im1, ax=ax[0], shrink=1)

im2 = ax[1].imshow(phi_pred_maduro[:,100,:].T, vmin=0, vmax=0.3, cmap='jet')
ax[1].set_title('Estimated Porosity')
ax[1].set_aspect('auto')
ax[1].set_yticks(depth_indices)
ax[1].set_yticklabels(depth_values[depth_indices])
fig.colorbar(im2, ax=ax[1], shrink=1)

im3 = ax[2].imshow(phi_mature[:,100,:].T, vmin=0, vmax=0.3, cmap='jet')
ax[2].set_title('Ground Truth Porosity')
ax[2].set_aspect('auto')
ax[2].set_yticks(depth_indices)
ax[2].set_yticklabels(depth_values[depth_indices])
fig.colorbar(im3, ax=ax[2], shrink=1)
fig.tight_layout()
        
fig.savefig("./plots/section_predicted_inline.png")
fig.savefig("./plots/section_predicted_inline.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Bloque de Exploración

In [ ]:
seis_normalized_exploration = (seis_exploration - np.min(seis_exploration))/(np.max(seis_exploration)-np.min(seis_exploration))

X_exploracion = seis_normalized_exploration.reshape(-1, 124)

X_exploracion = np.expand_dims(X_exploracion, axis=1)
phi_pred_exploracion = model_phi.predict(X_exploracion)
phi_pred_exploracion = inverse_min_max_scale(phi_pred_exploracion, phi_min, phi_max)

In [ ]:
seis_exploration.shape

In [ ]:
phi_pred_exploracion = phi_pred_exploracion.reshape(183, 370, 124)

Necesito extraer la sección exacta por donde pasa el pozo, para eso necesito conocer el rango e IL y XL, eso está en el archivo data prep de la sismica y tambien necesito conocer en que inline y xline estan los pozos para visualizarlos

Amplitude Inline range: 13 - 195

Amplitude Crossline range: 930 - 2140

In [ ]:
fig, ax = plt.subplots(1,3, figsize = (15, 4))

fig.suptitle('Inline 84 Wells CCS1 VW1')

im1 = ax[0].imshow(seis_exploration[84-13,:,:].T, cmap='Greys')
ax[0].set_title('Seismic')
ax[0].set_aspect('auto')
ax[0].set_yticks(depth_indices)
ax[0].set_yticklabels(depth_values[depth_indices])
fig.colorbar(im1, ax=ax[0], shrink=1)

im2 = ax[1].imshow(phi_pred_exploracion[84-13,:,:].T, vmin=0, vmax=0.3, cmap='jet')
ax[1].set_title('Estimated Porosity')
ax[1].set_aspect('auto')
ax[1].set_yticks(depth_indices)
ax[1].set_yticklabels(depth_values[depth_indices])
fig.colorbar(im2, ax=ax[1], shrink=1)

im3 = ax[2].imshow(phi_exploration[84-13,:,:].T, vmin=0, vmax=0.3, cmap='jet')
ax[2].set_title('Ground Truth Porosity')
ax[2].set_aspect('auto')
ax[2].set_yticks(depth_indices)
ax[2].set_yticklabels(depth_values[depth_indices])
fig.colorbar(im3, ax=ax[2], shrink=1)

fig.tight_layout()
fig.savefig("./plots/section_predicted_inline.pdf", format="pdf", bbox_inches="tight")

plt.show()
